# Pipeline di Estrazione Landmark (Val Set)
Questo notebook estrae i landmark spaziali mediapipe dai video raw del dataset ASL Citizen.
L'obiettivo è trasformare pesanti video `.mp4` in leggeri e informativi array `.npy` contenenti le coordinate (x,y,z) del corpo e delle mani.

### Fasi principali:
1. **Caricamento Dipendenze e Configurazione**: Setup dell'API MediaPipe Holistic.
2. **Estrazione Frame-by-Frame**: Lettura video e inferenza mediapipe per ogni frame.
3. **Salvataggio**: Export della matrice come array numpy compattato.

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import os
import pandas as pd
from tqdm import tqdm
import logging
from logging.handlers import QueueHandler, QueueListener
import multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed
from datetime import datetime
import time


## 1. Configurazione Percorsi
Impostiamo i path che puntano al file `val.csv` e alla cartella di output `Dataset_Keypoints_Val`.


In [ ]:
INPUT_BASE = 'data/raw/ASL_Citizen/videos'
CSV_PATH = 'data/raw/ASL_Citizen/splits/val.csv'
OUTPUT_BASE = 'data/processed/Dataset_Keypoints_Val'
NUM_WORKERS = max(1, multiprocessing.cpu_count() - 2) # Bilanciamento CPU


## 2. Funzioni Core di Estrazione
Definiamo la funzione `extract_keypoints` che converte un frame in un vettore 1D concatenando:
- 33 Pose landmarks (senza z) -> 66 valori
- 21 Face landmarks (ridotti) -> 66 valori
- 21 Left Hand landmarks -> 63 valori
- 21 Right Hand landmarks -> 63 valori

Totale = 258 feature spaziali per frame.

In [ ]:

# Configurazione percorsi
DATASET_DIR = 'data/raw/ASL_Citizen'
VIDEOS_DIR = os.path.join(DATASET_DIR, 'videos')
VAL_CSV_PATH = os.path.join(DATASET_DIR, 'splits', 'val.csv')
OUTPUT_BASE = "data/processed/Dataset_Keypoints_Val"

# Setup Logging
log_filename = f'extraction_val_log_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'
logging.basicConfig(
    filename=log_filename,
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

def extract_keypoints(results):
    """Estrae i 258 valori dal risultato di Holistic (identico al Train)"""
    pose = np.array([[res.x, res.y, res.z, res.visibility] for res in
                     results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(132)
    lh = np.array([[res.x, res.y, res.z] for res in
                   results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(63)
    rh = np.array([[res.x, res.y, res.z] for res in
                   results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(63)
    return np.concatenate([pose, lh, rh])


def process_video_worker(args):
    """Elabora un singolo video: estrae i landmark frame per frame e salva il risultato come file .npy."""
    video_filename, label, worker_id = args
    video_path = os.path.join(VIDEOS_DIR, video_filename)

    # Cartella di output per il segno specifico
    action_dir = os.path.join(OUTPUT_BASE, str(label).upper().strip())
    os.makedirs(action_dir, exist_ok=True)
    output_file = os.path.join(action_dir, video_filename.split('.')[0] + ".npy")

    # Se il file esiste già, saltiamo per permettere il resume
    if os.path.exists(output_file):
        return f"SKIP: {video_filename}"

    if not os.path.exists(video_path):
        return f"FAIL: {video_filename} non trovato"

    try:
        start_time = time.time()
        
        # Inizializzazione MediaPipe Holistic per questo worker
        mp_holistic = mp.solutions.holistic
        with mp_holistic.Holistic(static_image_mode=False, min_detection_confidence=0.5) as holistic:
            cap = cv2.VideoCapture(video_path)
            sequence = []
            frame_count = 0

            while cap.isOpened():
                ret, frame = cap.read()
                if not ret: break

                # Conversione BGR -> RGB per MediaPipe
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame.flags.writeable = False
                results = holistic.process(frame)
                sequence.append(extract_keypoints(results))
                frame_count += 1

            cap.release()

            elapsed = time.time() - start_time
            if len(sequence) > 0:
                np.save(output_file, np.array(sequence))
                log_msg = f"SUCCESS: {video_filename} | Label: {label} | Frames: {frame_count} | {elapsed:.1f}s"
                logging.info(log_msg)
                return log_msg
            else:
                logging.warning(f"EMPTY: {video_filename} non ha prodotto frame validi.")
                return f"EMPTY: {video_filename}"

    except Exception as e:
        logging.error(f"ERROR: {video_filename} | {str(e)}")
        return f"FAIL: {video_filename} | {str(e)}"




## 3. Esecuzione Multi-Processo (Runner)
Per processare migliaia di video in tempi brevi, utilizziamo Python `multiprocessing`. Ogni core elabora un video separato per aggirare il GIL di Python.

In [ ]:
# Esecuzione della pipeline
if __name__ == '__main__':
    main()